In [1]:
import time
import numpy as np
import pandas as pd
import EL
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import pickle
sns.set()


/Users/jiguangli/opt/anaconda3/envs/EL/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Replicating Asset Pricing Experiment in (Chib, Shin, Simoni, JRSSB, 2022)

- paper link: https://arxiv.org/pdf/2110.13531

We have real data on monthly excess returns from Jan 1974 to Dec 2018.

- Data available from R package: https://siddharthachib.org/rpackages/czfactor/czfactor.pdf
- Data has shape (540, 12)

**Moment Conditions in the Paper**

- Let $f_t = (x_t', w_t')' \in \mathbb{R}^{12}$ contain 12 variables "Mkt", "SMB", "HML"	"RMW",	"CMA"	"MOM", 	"IA", 	"ROE", 	"PEAD", "FIN",	"MGMT",	"PERF".
- Let $x_t \in \mathbb{R}^{k_x}$ be risk factors, $\beta \in \mathbb{R}^{k_x}$ risk factor premia, $\mu_x= E[x_t]$. We are interested in learning $(\beta, \mu_x)$.
-  We have the mixed moment conditions:
\begin{align}
\mathbb{E}[(1-\beta'(x_t - \mu_x))f_t] &= 0,  \\
\mathbb{E}[x_t - \mu_x | f_{t-1}] = 0,
\end{align}
where the first equation is uncoditional, and the second equation is condional moment restrictions.

**Experiment Configurations in the Paper**

- Let $x_t$ be excess return on the market portfolio ("Mkt"). So $(\beta, \mu_x)$ are scalars.
- Expanded moments:
    $$\mathbb{E}[(x_t-\mu_x) ] \otimes [q^K(f_{1, t-1}, \tilde{q}^K(f_{2, t-1}), \cdots, \tilde{q}^K(f_{12, t-1} )] = 0$$
- $q^K(f_{1, t-1})$ consits of $K=3$ basis, and $\tilde{q}^K(f_{j, t-1}$ for $(j \geq 2)$ has 2 basis finction derived from $q^K(f_{j, t-1})$ by subtracting the second and third columns from the first and then dropping the last. 
- Total conditions: 3 + 2*11 + 12 = 37.

In [2]:
def summarize_1d(x: np.ndarray) -> dict:
    x = np.asarray(x, float)
    return {
        "mean": float(np.mean(x)),
        "sd": float(np.std(x, ddof=1)),
        "median": float(np.median(x)),
        "q05": float(np.quantile(x, 0.05)),
        "q95": float(np.quantile(x, 0.95)),
    }


def print_summary(name: str, s: dict):
    print(
        f"{name:>6}: "
        f"mean={s['mean']:+.6f}, sd={s['sd']:.6f}, median={s['median']:+.6f}, "
        f"q05={s['q05']:+.6f}, q95={s['q95']:+.6f}"
    )


In [3]:
with open("ap_result.pkl", "rb") as f:
    model = pickle.load(f)

# Result Analysis

In [4]:
thetas = model.thetas
b_draws = thetas[:, 0]
mu_draws = thetas[:, 1]


print_summary("b", summarize_1d(b_draws))
print_summary("mu_x", summarize_1d(mu_draws))


theta_mean = np.mean(thetas, axis=0)
print("------------------------------------------------------------")
print(f"Posterior mean theta = [{theta_mean[0]:+.6f}, {theta_mean[1]:+.6f}]")
print("------------------------------------------------------------")


     b: mean=+2.822848, sd=0.802291, median=+2.918332, q05=+1.366509, q95=+4.249198
  mu_x: mean=+0.006169, sd=0.001665, median=+0.006147, q05=+0.003167, q95=+0.009118
------------------------------------------------------------
Posterior mean theta = [+2.822848, +0.006169]
------------------------------------------------------------
